# MizanIQ Phase-2 OCR/Vision Benchmark (Kaggle lab)

First experiment: **A** Tesseract CPU baseline (DEV-004, DEV-010), then **B** PaddleOCR-VL-1.6
(DEV-004/010, optionally 008/009). **C** Qwen3-VL on DEV-008/009 is OPTIONAL and OFF by default.

**Setup:** upload the export bundle (`inputs/`, `manifest.json`, `ocr_benchmark_truth.json`,
`PROMPTS.md`, `RESULT_SCHEMA.json`) — see `notebooks/README.md`. CPU suffices for A;
enable GPU (T4) for B/C. No weights or secrets are stored in this notebook.
Rule: never hand-edit model outputs before scoring.

In [ ]:
# ── 0. Environment information ──
import json
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

print('python:', platform.python_version(), '| impl:', platform.python_implementation())
print('platform:', platform.platform())
try:
    import torch
    print('torch:', torch.__version__, '| cuda_available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
except ImportError:
    print('torch: not installed (CPU-only session)')
print('nvidia-smi:', shutil.which('nvidia-smi') or 'absent')
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'], capture_output=True,
                         text=True).stdout.strip())

DEVICE = 'kaggle-GPU' if shutil.which('nvidia-smi') else 'kaggle-CPU'
print('DEVICE =', DEVICE)

## 1. Load benchmark

Validate the manifest and show the selected DEV IDs. Truth is loaded for field-name
alignment and local self-checks only — model outputs are never edited to match it.

In [ ]:
WORKSPACE = Path('.')
manifest = json.loads((WORKSPACE / 'manifest.json').read_text(encoding='utf-8'))
truth = json.loads((WORKSPACE / 'ocr_benchmark_truth.json').read_text(encoding='utf-8'))

print(f"{'document':10} {'role':10} {'tasks':14} file")
for entry in manifest['entries']:
    path = WORKSPACE / entry['workspace_file']
    assert path.is_file(), f'missing input: {path}'
    print(f"{entry['document_id']:10} {entry['role']:10} "
          f"{','.join(entry['tasks']):14} {entry['workspace_file']}")
print('\nselected A/B:', ['DEV-004', 'DEV-010'], '| optional B/C:', ['DEV-008', 'DEV-009'])

results = []  # standard-schema result dicts accumulate here

## 2. Common result utilities

Mirrors `src/ocrbench/schema.py` (the local validator is authoritative).

In [ ]:
PROMPTS = {
    'ocr': ('Transcribe all visible text faithfully, top to bottom. '
            'Preserve numbers, identifiers, and punctuation exactly as shown. '
            'For Arabic text, output the logical reading order. '
            'Do not infer, translate, or fill in missing values.'),
    'table': ('Recover each visible table as structured rows. '
              'Associate every row label with its visible numeric value exactly '
              'as printed. Preserve numbers and identifiers character-for-character. '
              'Do not calculate, round, or invent missing cells. '
              'If a cell is unreadable, write UNREADABLE for that cell only.'),
    'chart': ('Step 1 \u2014 report only what is visible: chart title, axis labels, '
              'every labeled value, and the legend. '
              'Step 2 \u2014 separately, describe the trend using only the visible '
              'values (direction, notable shifts, highest/lowest point). '
              'Do not estimate values that are not labeled.'),
    'kpi': ('Recover each displayed KPI card label and its value exactly as shown, '
            'then each bar label and value, then the budget panel figures. '
            'Separately, quote the visible budget-status statement verbatim. '
            'Do not recompute variances or reinterpret the verdict.'),
}


def blank_result(model, model_version, document_id):
    return {'model': model, 'model_version': model_version, 'device': DEVICE,
            'document_id': document_id, 'latency_ms': 0.0, 'text': '',
            'fields': {}, 'tables': [], 'visual_description': '', 'warnings': []}


def check_result(result):
    required = {'model': str, 'model_version': str, 'device': str,
                'document_id': str, 'latency_ms': (int, float), 'text': str,
                'fields': dict, 'tables': list, 'visual_description': str,
                'warnings': list}
    errors = [f'missing: {k}' for k in required if k not in result]
    errors += [f'type: {k}' for k, t in required.items()
               if k in result and not isinstance(result[k], t)]
    return errors

## 3. Experiment A — Tesseract baseline (DEV-004, DEV-010)

CPU only. Uses the `tesseract` CLI directly (no wrapper dependency). If Tesseract is
unavailable, results record the fact and the notebook continues — no unsafe workarounds.

In [ ]:
TESS_DOCS = ['DEV-004', 'DEV-010']
TESS_LANG = 'ara+eng'  # fallback to 'eng' below if the ara pack is absent


def tesseract_version():
    try:
        out = subprocess.run(['tesseract', '--version'], capture_output=True,
                             text=True, timeout=30).stdout
        return out.splitlines()[0] if out else 'unknown'
    except Exception:
        return 'unavailable'


def tesseract_langs():
    try:
        out = subprocess.run(['tesseract', '--list-langs'], capture_output=True,
                             text=True, timeout=30).stdout
        return [line.strip() for line in out.splitlines()[1:] if line.strip()]
    except Exception:
        return []


TESS_AVAILABLE = shutil.which('tesseract') is not None
TESS_VERSION = tesseract_version() if TESS_AVAILABLE else 'unavailable'
langs = tesseract_langs() if TESS_AVAILABLE else []
lang = TESS_LANG if all(p in langs for p in ('ara', 'eng')) else 'eng'
print('tesseract:', TESS_VERSION, '| langs:', langs or 'n/a', '| using:', lang)

for doc_id in TESS_DOCS:
    entry = next(e for e in manifest['entries'] if e['document_id'] == doc_id)
    result = blank_result('tesseract', TESS_VERSION, doc_id)
    if not TESS_AVAILABLE:
        result['warnings'] = ['tesseract binary unavailable in this environment']
    else:
        if lang != TESS_LANG:
            result['warnings'] = [f'ara traineddata missing; fell back to {lang}']
        started = time.perf_counter()
        try:
            proc = subprocess.run(
                ['tesseract', str(WORKSPACE / entry['workspace_file']),
                 'stdout', '-l', lang],
                capture_output=True, text=True, timeout=600)
            result['text'] = proc.stdout
            if proc.returncode != 0:
                result['warnings'].append(f'tesseract stderr: {proc.stderr[:300]}')
        except Exception as exc:
            result['warnings'].append(f'tesseract failed: {type(exc).__name__}')
        result['latency_ms'] = (time.perf_counter() - started) * 1000.0
    result['warnings'].append(f'lang config: {lang}; output NOT hand-corrected')
    assert not check_result(result), check_result(result)
    results.append(result)
    print(doc_id, f"{result['latency_ms']:.0f} ms", f'{len(result["text"])} chars',
          result['warnings'])

## 4. Experiment B — PaddleOCR-VL-1.6 (DEV-004, DEV-010, optionally 008/009)

Architect-locked candidate: **PaddleOCR-VL-1.6** — do NOT silently substitute another
version. If 1.6 cannot run here, the cell records the blocker and continues; the version
is reported, never fudged. Fill `run_paddleocr_vl` from the official PaddleOCR-VL-1.6
inference docs; GPU recommended.

In [ ]:
PADDLE_MODEL_ID = 'PaddleOCR-VL-1.6'  # locked; report blockers, never substitute
PADDLE_DOCS = ['DEV-004', 'DEV-010']
PADDLE_DOCS_OPTIONAL = ['DEV-008', 'DEV-009']
RUN_PADDLE_OPTIONAL = False  # set True after the primary pair scores

try:
    import paddleocr
    PADDLE_PKG = getattr(paddleocr, '__version__', 'unknown')
except ImportError:
    PADDLE_PKG = 'not-installed'
print('paddleocr package:', PADDLE_PKG)


def run_paddleocr_vl(image_path):
    """Return (text, fields, tables, visual_description) for one image.

    Implement from the official PaddleOCR-VL-1.6 inference docs inside
    this cell on Kaggle. Keep fields keys aligned with ocr_benchmark_truth.
    """
    raise NotImplementedError(
        'PADDLE_BLOCKER: fill run_paddleocr_vl from official PaddleOCR-VL-1.6 docs')


for doc_id in PADDLE_DOCS + (PADDLE_DOCS_OPTIONAL if RUN_PADDLE_OPTIONAL else []):
    entry = next(e for e in manifest['entries'] if e['document_id'] == doc_id)
    result = blank_result('paddleocr-vl', PADDLE_MODEL_ID, doc_id)
    result['warnings'].append(f'package: paddleocr=={PADDLE_PKG}')
    if PADDLE_PKG == 'not-installed':
        result['warnings'].append('PADDLE_BLOCKER: paddleocr not installed')
    else:
        started = time.perf_counter()
        try:
            text, fields, tables, visual = run_paddleocr_vl(
                WORKSPACE / entry['workspace_file'])
            result.update({'text': text, 'fields': fields,
                           'tables': tables, 'visual_description': visual})
        except NotImplementedError as exc:
            result['warnings'].append(str(exc))
        except Exception as exc:
            result['warnings'].append(f'inference failed: {type(exc).__name__}: {exc}')
        result['latency_ms'] = (time.perf_counter() - started) * 1000.0
    assert not check_result(result), check_result(result)
    results.append(result)
    print(doc_id, f"{result['latency_ms']:.0f} ms", result['warnings'])

## 5. Experiment C — Qwen3-VL (OPTIONAL, DEV-008/009 visual understanding)

NOT required for the first run. Prefer `Qwen/Qwen3-VL-8B-Instruct` when the GPU runs it
reliably; use the 4B fallback only when documented. Never change size mid-comparison
without recording it. Keep extraction (labels/values) and interpretation (trend/verdict)
in separate outputs per the chart/KPI prompts.

In [ ]:
RUN_EXPERIMENT_C = False  # flip to True for the visual-reasoning round
QWEN_MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'  # fallback (record!): Qwen/Qwen3-VL-4B-Instruct
QWEN_DOCS = ['DEV-008', 'DEV-009']

if not RUN_EXPERIMENT_C:
    print('Experiment C skipped (optional; set RUN_EXPERIMENT_C=True to run)')
else:
    try:
        import transformers
        print('transformers:', transformers.__version__)
    except ImportError:
        print('QWEN_BLOCKER: transformers not installed')
    # Implement Qwen3-VL-Instruct inference here from official docs:
    # load QWEN_MODEL_ID, apply the chart prompt to DEV-008 and the KPI
    # prompt to DEV-009 (PROMPTS above), record per-doc latency, append
    # blank_result('qwen3-vl', QWEN_MODEL_ID, doc_id) dicts to results.
    print('TODO: Qwen3-VL inference (see cell markdown + PROMPTS)')

## 6. Export

Write `kaggle_results.json` + `experiment_meta.json`. Both are validated against the
result contract before saving. Download both to local `data/benchmark/results/`.

In [ ]:
errors = {r['document_id']: check_result(r) for r in results}
bad = {k: v for k, v in errors.items() if v}
assert not bad, bad
with open('kaggle_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
meta = {'models': sorted({(r['model'], r['model_version']) for r in results}),
        'device': DEVICE, 'n_results': len(results),
        'exported_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
with open('experiment_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"{'document':10} {'model':14} {'ms':>9}  warnings")
for r in results:
    print(f"{r['document_id']:10} {r['model']:14} {r['latency_ms']:9.0f}  "
          f"{'; '.join(r['warnings'])}")
print('\nsaved kaggle_results.json + experiment_meta.json')